# microWakeWord — Nyra EN V1 SYNTHETIC (66 GB free-disk profile)

This Colab notebook trains the English wake word **"Nyra"**, pronounced by default as **"NYE-ruh"** (approximately `/ˈnaɪrə/`).

The technical model ID is **`nyra_en`**.

## What this version does

- Uses **no real recordings**.
- Uses the English **LibriTTS-R multi-speaker** Piper sample-generator model.
- Generates a **small preview set first**.
- Lets you **listen to every preview sample directly in Colab**.
- Requires you to type **`OK`** before full positive-sample generation can continue.
- Generates synthetic hard confusable negatives tailored to English **Nyra / NYE-ruh**.
- Uses the standard `speech`, `no_speech`, `dinner_party`, and `dinner_party_eval` feature datasets.
- Exports `nyra_en.tflite` and `nyra_en.json` for ESPHome microWakeWord.

## Disk profile

Designed around a Colab runtime with approximately:

- 113 GB total local disk
- ~66 GB initially free
- ~52 GB system RAM
- NVIDIA L4 GPU

The notebook deletes source WAVs after their mmap features have been generated to protect disk space.

## Important

The pronunciation is controlled by `WAKE_WORD_PHONEMES`. The default is the English **NYE-ruh** reading. Do not start the full generation unless the preview actually sounds like the Nyra you want.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    NYRA EN V1 SYNTHETIC CONFIG                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

WAKE_WORD = "Nyra"
OUTPUT_NAME = "nyra_en"
AUTHOR = "Nicola"
AUTHOR_WEBSITE = ""

MODE = "generate"
DRIVE_FOLDER = "wakeword_training_nyra_en"

# -----------------------------------------------------------------------
# Pronunciation
# -----------------------------------------------------------------------
# Default target: English "NYE-ruh", approximately /ˈnaɪrə/.
#
# piper-sample-generator accepts phoneme input. This avoids asking the
# English TTS model to guess how the invented spelling "Nyra" should sound.
WAKE_WORD_PHONEMES = "nˈaɪɹə"

# Optional alternative if you ever want "NEER-uh":
# WAKE_WORD_PHONEMES = "nˈɪɹə"

# -----------------------------------------------------------------------
# Preview gate
# -----------------------------------------------------------------------
PREVIEW_SAMPLES = 24
PREVIEW_APPROVAL_TEXT = "OK"

# -----------------------------------------------------------------------
# Full synthetic positives
# -----------------------------------------------------------------------
# The LibriTTS-R generator is highly multi-speaker, so a fully synthetic
# English model can benefit from a large positive set.
POSITIVE_SAMPLES = 24000

# -----------------------------------------------------------------------
# Hard negatives
# -----------------------------------------------------------------------
# Words / short expressions intentionally close to NYE-ruh.
# These are NEGATIVE examples: the model should learn not to fire on them.
CONFUSABLE_PHONEMES = [
    ("Myra",    "mˈaɪɹə"),
    ("Lyra",    "lˈaɪɹə"),
    ("Nyla",    "nˈaɪlə"),
    ("Tyra",    "tˈaɪɹə"),
    ("Mira",    "mˈɪɹə"),
    ("Nina",    "nˈinə"),
    ("Nora",    "nˈɔɹə"),
    ("Ira",     "ˈaɪɹə"),
    ("Sierra",  "siˈɛɹə"),
    ("near a",  "nˈɪɹə"),
    ("fire",    "fˈaɪɹ"),
    ("higher",  "hˈaɪɹ"),
]
SAMPLES_PER_CONFUSABLE = 700

# Feature repetitions. 2 keeps the 66 GB profile practical.
TRAIN_REPETITION = 2

# Never intentionally consume the final 8 GB of local runtime disk.
DISK_RESERVE_GB = 8.0

# ESPHome manifest starting values.
# Tune these later on the real ESP Audio board if necessary.
PROBABILITY_CUTOFF = 0.70
SLIDING_WINDOW_SIZE = 3
TENSOR_ARENA_SIZE = 50000
TRAINED_LANGUAGES = ["en"]

print(f"Training {WAKE_WORD!r} -> {OUTPUT_NAME}.tflite")
print(f"Target pronunciation: NYE-ruh")
print(f"Positive phonemes: {WAKE_WORD_PHONEMES}")
print(f"Preview samples: {PREVIEW_SAMPLES}")
print(f"Full positive samples: {POSITIVE_SAMPLES}")
print(f"Hard negatives: {len(CONFUSABLE_PHONEMES)} × {SAMPLES_PER_CONFUSABLE}")
print(f"Training feature repetition: {TRAIN_REPETITION}")
print(f"Disk reserve: {DISK_RESERVE_GB:.1f} GB")


In [ ]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


In [ ]:
# === Runtime sanity + disk guard helpers ===
import os, shutil, subprocess
from pathlib import Path

os.chdir("/content")

def disk_stats(path="/content"):
    total, used, free = shutil.disk_usage(path)
    gb = 1024**3
    return total/gb, used/gb, free/gb

def show_disk(label="disk"):
    total, used, free = disk_stats()
    print(f"[{label}] total={total:.1f} GB used={used:.1f} GB free={free:.1f} GB")
    return free

def require_free(min_free_gb, label="operation"):
    free = show_disk(label)
    if free < min_free_gb:
        raise RuntimeError(
            f"Not enough disk for {label}: {free:.1f} GB free, "
            f"need at least {min_free_gb:.1f} GB."
        )

def dir_gb(path):
    p = Path(path)
    if not p.exists():
        return 0.0
    result = subprocess.run(
        ["du", "-sb", str(p)],
        capture_output=True, text=True, check=True
    )
    return int(result.stdout.split()[0]) / (1024**3)

# Clean only working data for THIS run if Run All is repeated.
for d in [
    "/content/generated_samples",
    "/content/confusable_negatives",
    "/content/generated_augmented_features",
    "/content/confusable_features",
    "/content/negative_datasets",
    f"/content/trained_models/{OUTPUT_NAME}",
]:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

os.makedirs("/content/generated_samples", exist_ok=True)
os.makedirs("/content/confusable_negatives", exist_ok=True)
os.makedirs("/content/negative_datasets", exist_ok=True)

initial_free = show_disk("initial")
if initial_free < 55:
    print(
        "WARNING: this notebook was designed around ~66 GB free. "
        "It will continue, but disk guards may stop it before training."
    )


In [ ]:
# === Piper sample generator install ===
import glob, os, shutil, subprocess, sys, urllib.request, importlib

PIPER_REPO_DIR = "/content/piper"
PIPER_SAMPLE_GENERATOR_DIR = "/content/piper-sample-generator"

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "espeak-ng"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "pip", "setuptools", "wheel", "cython"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade",
     "piper-tts", "piper-sample-generator"],
    check=True,
)

if not os.path.exists(PIPER_REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/rhasspy/piper", PIPER_REPO_DIR],
        check=True,
    )

if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/rhasspy/piper-sample-generator",
         PIPER_SAMPLE_GENERATOR_DIR],
        check=True,
    )

# Build Piper monotonic_align extension
PIPER_PYTHON_DIR = f"{PIPER_REPO_DIR}/src/python"
MA_DIR = f"{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align"
MA_IMPORT_DIR = f"{MA_DIR}/monotonic_align"
MA_BUILD_DIR = f"{MA_DIR}/piper_train/vits/monotonic_align"

shutil.rmtree(f"{PIPER_PYTHON_DIR}/build", ignore_errors=True)
shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
shutil.rmtree(f"{MA_DIR}/piper_train", ignore_errors=True)

os.makedirs(MA_IMPORT_DIR, exist_ok=True)
os.makedirs(MA_BUILD_DIR, exist_ok=True)
open(f"{MA_IMPORT_DIR}/__init__.py", "a").close()

subprocess.run(
    f"cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace",
    shell=True,
    check=True,
)

built = next(iter(glob.glob(f"{MA_BUILD_DIR}/core.*")), None)
assert built, "monotonic_align core extension build failed"
shutil.copy2(built, MA_IMPORT_DIR)

for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

importlib.invalidate_caches()

# English multi-speaker synthetic voice model used by piper-sample-generator.
# Phoneme input explicitly controls how the invented name Nyra is pronounced.
MODEL_PATH = "/content/models/en_US-libritts_r-medium.pt"
MODEL_CONFIG_PATH = f"{MODEL_PATH}.json"
os.makedirs("/content/models", exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print("Downloading Piper sample-generator model...")
    urllib.request.urlretrieve(
        "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",
        MODEL_PATH,
    )
    urllib.request.urlretrieve(
        "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json",
        MODEL_CONFIG_PATH,
    )

# Environment inherited by ALL Piper subprocesses.
PIPER_ENV = os.environ.copy()
PIPER_ENV["PYTHONPATH"] = ":".join([
    PIPER_PYTHON_DIR,
    PIPER_SAMPLE_GENERATOR_DIR,
    PIPER_ENV.get("PYTHONPATH", ""),
])

# Smoke test: fail here rather than after sample generation begins.
test = subprocess.run(
    [sys.executable, "-c",
     "import piper_train; import piper_sample_generator; print('Piper imports OK')"],
    env=PIPER_ENV,
    capture_output=True,
    text=True,
)
print(test.stdout)
if test.returncode != 0:
    print(test.stderr)
    test.check_returncode()

print("Piper ready")


In [ ]:
# === Preview Nyra pronunciation, listen, APPROVE, then generate positives ===
import os, subprocess, sys, shutil, glob
from IPython.display import Audio, display, Markdown

PIPER_BATCH = 256

def run_piper_phonemes(target_phonemes, max_samples, output_dir, batch_size=PIPER_BATCH):
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        sys.executable,
        "-m", "piper_sample_generator",
        target_phonemes,
        "--phoneme-input",
        "--model", MODEL_PATH,
        "--max-samples", str(max_samples),
        "--batch-size", str(batch_size),
        "--noise-scales", "0.5",
        "--noise-scale-ws", "0.6",
        "--output-dir", output_dir,
    ]

    proc = subprocess.run(
        cmd,
        env=PIPER_ENV,
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        proc.check_returncode()


# -----------------------------------------------------------------------
# 1) Generate a SMALL preview only
# -----------------------------------------------------------------------
PREVIEW_DIR = "/content/nyra_en_preview"
shutil.rmtree(PREVIEW_DIR, ignore_errors=True)
os.makedirs(PREVIEW_DIR, exist_ok=True)

print(f"Generating {PREVIEW_SAMPLES} preview samples for Nyra (NYE-ruh)...")
run_piper_phonemes(
    WAKE_WORD_PHONEMES,
    PREVIEW_SAMPLES,
    PREVIEW_DIR,
    batch_size=min(PIPER_BATCH, PREVIEW_SAMPLES),
)

preview_files = sorted(glob.glob(f"{PREVIEW_DIR}/*.wav"))
assert preview_files, "No preview WAVs were generated."

print()
print(f"Generated {len(preview_files)} preview WAVs.")
print("Listen to them below. Pay attention to:")
print("  • NYE-ruh pronunciation")
print("  • first syllable = 'nye' /naɪ/")
print("  • second syllable = reduced 'ruh'")
print("  • no accidental 'NEER-uh', 'NIRA', or 'NAY-ra'")
print()

for i, wav in enumerate(preview_files, 1):
    display(Markdown(f"**Preview {i:02d}/{len(preview_files)}** — `{os.path.basename(wav)}`"))
    display(Audio(wav))


# -----------------------------------------------------------------------
# 2) Mandatory human approval
# -----------------------------------------------------------------------
print()
print("============================================================")
print("DO NOT APPROVE unless the preview sounds like English NYE-ruh.")
print(f'Type exactly {PREVIEW_APPROVAL_TEXT!r} to continue.')
print("Anything else stops the notebook BEFORE the expensive generation.")
print("============================================================")

approval = input("Approval: ").strip()

if approval != PREVIEW_APPROVAL_TEXT:
    raise RuntimeError(
        "Synthetic pronunciation NOT approved. "
        "Full generation and training were intentionally stopped."
    )

print("✓ Pronunciation approved.")


# -----------------------------------------------------------------------
# 3) Full positive generation
# -----------------------------------------------------------------------
require_free(DISK_RESERVE_GB + 4, "positive sample generation")

shutil.rmtree("/content/generated_samples", ignore_errors=True)
os.makedirs("/content/generated_samples", exist_ok=True)

print()
print(f"Generating {POSITIVE_SAMPLES} English Nyra positives...")
run_piper_phonemes(
    WAKE_WORD_PHONEMES,
    POSITIVE_SAMPLES,
    "/content/generated_samples",
)

n = sum(
    1 for f in os.listdir("/content/generated_samples")
    if f.endswith(".wav")
)

print(f"Total positive samples: {n}")
print(f"Positive WAV size: {dir_gb('/content/generated_samples'):.2f} GB")
show_disk("after positive WAVs")

assert n >= POSITIVE_SAMPLES * 0.95, "Too few positive samples generated"

# Preview is no longer needed.
shutil.rmtree(PREVIEW_DIR, ignore_errors=True)
print("✓ Preview WAVs removed after approval.")


In [ ]:
# === Generate English hard confusable negatives ===
import os, re, shutil
from pathlib import Path

os.makedirs("/content/confusable_negatives", exist_ok=True)

for i, (label, phonemes) in enumerate(CONFUSABLE_PHONEMES, 1):
    require_free(DISK_RESERVE_GB + 2, f"confusable {label}")

    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", label.lower()).strip("_")
    tmp = f"/tmp/confusable_{safe}"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp, exist_ok=True)

    print(
        f"[{i}/{len(CONFUSABLE_PHONEMES)}] "
        f"{label!r}: {SAMPLES_PER_CONFUSABLE} samples ({phonemes})"
    )

    run_piper_phonemes(
        phonemes,
        SAMPLES_PER_CONFUSABLE,
        tmp,
    )

    moved = 0
    for filename in os.listdir(tmp):
        if filename.endswith(".wav"):
            os.rename(
                f"{tmp}/{filename}",
                f"/content/confusable_negatives/{safe}_{filename}",
            )
            moved += 1

    shutil.rmtree(tmp, ignore_errors=True)
    print(f"  ✓ moved {moved}")

n = sum(
    1 for f in os.listdir("/content/confusable_negatives")
    if f.endswith(".wav")
)
print(f"Total hard-negative WAVs: {n}")
print(f"Hard-negative WAV size: {dir_gb('/content/confusable_negatives'):.2f} GB")
show_disk("after confusable WAVs")


In [ ]:
# === Standard negative FEATURE datasets: sequential + guarded ===
# These are pre-generated mmap/spectrogram feature datasets from kahrendt/microwakeword.
# We download ONE ZIP at a time, extract it, then immediately delete the ZIP.
#
# Required for FULL run:
#   speech
#   no_speech
#   dinner_party
#   dinner_party_eval
#
# The disk guard prevents filling the 113 GB Colab filesystem.
import os, zipfile, shutil
from huggingface_hub import hf_hub_download

BASE = "/content/negative_datasets"
REPO = "kahrendt/microwakeword"
STANDARD_DATASETS = [
    "speech",
    "no_speech",
    "dinner_party",
    "dinner_party_eval",
]

os.makedirs(BASE, exist_ok=True)

def download_extract_feature_dataset(name):
    out_dir = f"{BASE}/{name}"

    if os.path.exists(out_dir) and os.listdir(out_dir):
        print(f"{name}: already present ({dir_gb(out_dir):.2f} GB)")
        return

    # Keep a healthy buffer before we even start the download.
    require_free(DISK_RESERVE_GB + 8, f"download {name}")

    zip_name = f"{name}.zip"
    print(f"\n=== {name}: downloading {zip_name} ===")

    zip_path = hf_hub_download(
        repo_id=REPO,
        repo_type="dataset",
        filename=zip_name,
        local_dir=BASE,
    )

    zip_size = os.path.getsize(zip_path) / (1024**3)
    print(f"ZIP size: {zip_size:.2f} GB")
    show_disk(f"{name} downloaded")

    # Need at least reserve + ZIP size again as a conservative extraction buffer.
    require_free(
        DISK_RESERVE_GB + max(4.0, zip_size * 1.5),
        f"extract {name}",
    )

    print(f"Extracting {name}...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(BASE)

    # Delete archive immediately.
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # HF may leave cache blobs; remove only the local download cache for this BASE.
    cache_dir = os.path.join(BASE, ".cache")
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir, ignore_errors=True)

    assert os.path.exists(out_dir), f"{name} extraction did not create {out_dir}"

    print(f"✓ {name}: {dir_gb(out_dir):.2f} GB extracted")
    show_disk(f"after {name}")

    # Hard stop while there is still enough space to recover cleanly.
    if disk_stats()[2] < DISK_RESERVE_GB:
        raise RuntimeError(
            f"Disk reserve violated after {name}; stopped before further writes."
        )

for name in STANDARD_DATASETS:
    download_extract_feature_dataset(name)

print("\nStandard negative feature sets ready:")
for name in STANDARD_DATASETS:
    print(f"  {name:18s} {dir_gb(f'{BASE}/{name}'):.2f} GB")

show_disk("standard negatives complete")


In [ ]:
# === Positive + confusable feature extraction (disk-optimized) ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os, shutil, traceback

# Rich self-contained augmentation. We deliberately avoid raw FMA/AudioSet corpora:
# the standard speech/no_speech/dinner_party feature sets provide the extra negatives,
# while raw background corpora would consume the disk needed for mmap features.
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.20,
        "TanhDistortion": 0.10,
        "PitchShift": 0.20,
        "BandStopFilter": 0.10,
        "AddColorNoise": 0.35,
        "Gain": 1.00,
        "GainTransition": 0.30,
    },
    impulse_paths=[],
    background_paths=[],
    background_min_snr_db=-5,
    background_max_snr_db=20,
    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

SPLIT_CONFIG = {
    "training": {
        "split_name": "train",
        "repetition": TRAIN_REPETITION,
        "slide_frames": 10,
    },
    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },
    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}

def generate_feature_set(input_dir, output_root, label):
    clips = Clips(
        input_directory=input_dir,
        file_pattern="*.wav",
        max_clip_duration_s=None,
        remove_silence=True,
        random_split_seed=42,
        split_count=0.1,
    )

    os.makedirs(output_root, exist_ok=True)

    for split, cfg in SPLIT_CONFIG.items():
        require_free(DISK_RESERVE_GB + 3, f"{label} {split} features")

        out = f"{output_root}/{split}"
        mmap = f"{out}/wakeword_mmap"

        if os.path.exists(mmap):
            shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)

        print(
            f"Generating {label} {split} "
            f"(rep={cfg['repetition']}, slide={cfg['slide_frames']})..."
        )

        try:
            sg = SpectrogramGeneration(
                clips=clips,
                augmenter=augmenter,
                slide_frames=cfg["slide_frames"],
                step_ms=10,
            )

            RaggedMmap.from_generator(
                out_dir=mmap,
                batch_size=200,
                verbose=True,
                sample_generator=sg.spectrogram_generator(
                    split=cfg["split_name"],
                    repeat=cfg["repetition"],
                ),
            )
        except Exception:
            traceback.print_exc()
            shutil.rmtree(mmap, ignore_errors=True)
            raise

        print(
            f"  {label}/{split}: "
            f"{dir_gb(out):.2f} GB; free={disk_stats()[2]:.1f} GB"
        )

# 1) positives
generate_feature_set(
    "/content/generated_samples",
    "/content/generated_augmented_features",
    "positive",
)
print(f"✓ positive features: {dir_gb('/content/generated_augmented_features'):.2f} GB")

# Reclaim positive WAVs BEFORE generating confusable features.
shutil.rmtree("/content/generated_samples", ignore_errors=True)
print("Deleted positive source WAVs")
show_disk("after positive WAV cleanup")

# 2) hard confusables
generate_feature_set(
    "/content/confusable_negatives",
    "/content/confusable_features",
    "confusable",
)
print(f"✓ confusable features: {dir_gb('/content/confusable_features'):.2f} GB")

shutil.rmtree("/content/confusable_negatives", ignore_errors=True)
print("Deleted confusable source WAVs")

print("\nFeature extraction complete:")
for p in [
    "/content/generated_augmented_features",
    "/content/confusable_features",
    "/content/negative_datasets",
]:
    print(f"  {p}: {dir_gb(p):.2f} GB")
show_disk("before training config")

if disk_stats()[2] < DISK_RESERVE_GB:
    raise RuntimeError("Disk reserve violated before training")


In [ ]:
# === FULL training config YAML ===
import yaml, os
from pathlib import Path

required_paths = [
    "/content/generated_augmented_features/training/wakeword_mmap",
    "/content/confusable_features/training/wakeword_mmap",
    "/content/negative_datasets/speech",
    "/content/negative_datasets/no_speech",
    "/content/negative_datasets/dinner_party",
    "/content/negative_datasets/dinner_party_eval",
]
for p in required_paths:
    assert Path(p).exists(), f"Required feature set missing: {p}"

config = {
    "window_step_ms": 10,
    "train_dir": f"/content/trained_models/{OUTPUT_NAME}",

    "features": [
        dict(
            features_dir="/content/generated_augmented_features",
            sampling_weight=8.0,
            penalty_weight=2.0,
            truth=True,
            truncation_strategy="truncate_start",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/speech",
            sampling_weight=10.0,
            penalty_weight=2.5,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/dinner_party",
            sampling_weight=15.0,
            penalty_weight=3.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
        dict(
            features_dir="/content/negative_datasets/no_speech",
            sampling_weight=5.0,
            penalty_weight=1.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),

        # sampling_weight=0 means this is for ambient evaluation, not training.
        dict(
            features_dir="/content/negative_datasets/dinner_party_eval",
            sampling_weight=0.0,
            penalty_weight=1.0,
            truth=False,
            truncation_strategy="split",
            type="mmap",
        ),

        dict(
            features_dir="/content/confusable_features",
            sampling_weight=10.0,
            penalty_weight=6.0,
            truth=False,
            truncation_strategy="random",
            type="mmap",
        ),
    ],

    "training_steps": [25000, 20000],
    "positive_class_weight": [2, 2],
    "negative_class_weight": [40, 50],
    "learning_rates": [0.001, 0.0001],

    # 52 GB RAM + L4 profile. Still conservative enough to avoid needless risk.
    "batch_size": 128,

    "time_mask_max_size": [5, 5],
    "time_mask_count": [1, 1],
    "freq_mask_max_size": [3, 3],
    "freq_mask_count": [1, 1],

    "eval_step_interval": 500,
    "clip_duration_ms": 1500,

    # Restore the useful ambient false-positive objective.
    "target_minimization": 0.4,
    "minimization_metric": "ambient_false_positives_per_hour",
    "maximization_metric": "average_viable_recall",
}

os.makedirs(f"/content/trained_models/{OUTPUT_NAME}", exist_ok=True)

with open("/content/training_parameters.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("FULL training_parameters.yaml ready")
print(f"Feature sets: {len(config['features'])}")
print(f"Total steps: {sum(config['training_steps'])}")
print(f"Batch size: {config['batch_size']}")
show_disk("training config ready")


In [ ]:
# === Train the FULL model ===
import os, sys, subprocess, shutil

require_free(DISK_RESERVE_GB, "training start")

train_dir = f"/content/trained_models/{OUTPUT_NAME}"
shutil.rmtree(train_dir, ignore_errors=True)

env = os.environ.copy()
env["PYTHONPATH"] = "/content/microWakeWord:" + env.get("PYTHONPATH", "")
env["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"

cmd = [
    sys.executable,
    "-m", "microwakeword.model_train_eval",
    "--training_config", "/content/training_parameters.yaml",
    "--train", "1",
    "--restore_checkpoint", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
]

print("Running FULL training:")
print(" ".join(cmd))
print()
show_disk("training launch")

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
print()
print("Exit code:", proc.returncode)

assert proc.returncode == 0, (
    "Training failed. Do NOT regenerate feature data; inspect the output above."
)

print("✓ FULL training completed")
show_disk("after training")


In [ ]:
# === Verify final model ===
import os

root = f"/content/trained_models/{OUTPUT_NAME}"

os.system(
    f'find "{root}" -type f '
    r'\( -name "*.tflite" -o -name "*.h5" -o -name "*.keras" -o -name "*.ckpt*" \) '
    '-exec ls -lh {} \\; | tail -30'
)

tflite_src = (
    f"{root}/tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

assert os.path.exists(tflite_src), (
    f"Final quantized TFLite not found at {tflite_src}"
)

print()
print("✓ Final quantized model found")
print(f"Size: {os.path.getsize(tflite_src) / 1024:.1f} KB")


In [ ]:
# === Export + automatic Drive backup ===
import os, json, shutil, datetime

tflite_src = (
    f"/content/trained_models/{OUTPUT_NAME}/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)
assert os.path.exists(tflite_src), f"No model at {tflite_src}"

OUT_TFLITE = f"/content/{OUTPUT_NAME}.tflite"
OUT_JSON = f"/content/{OUTPUT_NAME}.json"

shutil.copy2(tflite_src, OUT_TFLITE)

manifest = {
    "type": "micro",
    "wake_word": WAKE_WORD,
    "author": AUTHOR,
    "model": f"{OUTPUT_NAME}.tflite",
    "trained_languages": TRAINED_LANGUAGES,
    "version": 2,
    "micro": {
        "probability_cutoff": PROBABILITY_CUTOFF,
        "feature_step_size": 10,
        "sliding_window_size": SLIDING_WINDOW_SIZE,
        "tensor_arena_size": TENSOR_ARENA_SIZE,
        "minimum_esphome_version": "2024.7.0",
    },
}
if AUTHOR_WEBSITE.strip():
    manifest["website"] = AUTHOR_WEBSITE.strip()

with open(OUT_JSON, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"TFLite: {os.path.getsize(OUT_TFLITE)/1024:.1f} KB")
print(json.dumps(manifest, indent=2))

artifacts = [
    (OUT_TFLITE, f"{OUTPUT_NAME}.tflite"),
    (OUT_JSON, f"{OUTPUT_NAME}.json"),
    (
        f"/content/trained_models/{OUTPUT_NAME}/best_weights.weights.h5",
        f"{OUTPUT_NAME}_best_weights.weights.h5",
    ),
    (
        "/content/training_parameters.yaml",
        f"{OUTPUT_NAME}_training_parameters.yaml",
    ),
]

for src_path, dest_name in artifacts:
    assert os.path.exists(src_path), f"Backup source missing: {src_path}"
    dest = f"{DRIVE_DIR}/{dest_name}"
    shutil.copy2(src_path, dest)
    print(f"pushed -> {dest}")

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f"{DRIVE_DIR}/_run_finished.txt", "w") as f:
    f.write(
        f"Nyra EN synthetic training finished at {ts}\n"
        f"Wake word: {WAKE_WORD}\n"
        f"Training repetition: {TRAIN_REPETITION}\n"
        f"Datasets: speech,no_speech,dinner_party,dinner_party_eval,confusables\n"
        f"Output: {OUTPUT_NAME}.tflite\n"
    )

print()
print("==============================================")
print("DONE — NYRA EN V1 SYNTHETIC IS SAVED TO GOOGLE DRIVE")
print("==============================================")
print(DRIVE_DIR)


In [ ]:
# === Final status / disk usage ===
import os

print("Drive artifacts:")
os.system(f'ls -lh "{DRIVE_DIR}"')

print()
print("Runtime disk:")
os.system("df -h /content | tail -1")


## Deploy to ESPHome

After the run finishes, copy:

- `nyra_en.tflite`
- `nyra_en.json`

from:

`MyDrive/wakeword_training_nyra_en/`

to your ESPHome wake-word folder.

This model is trained with:

- English synthetic **Nyra = NYE-ruh** positives;
- English hard phonetic confusables;
- generic `speech`;
- generic `no_speech`;
- `dinner_party`;
- `dinner_party_eval` for ambient false-positive evaluation.

Initial manifest sensitivity:

- probability cutoff: **0.70**
- sliding window: **3**

Because **Nyra** is a short two-syllable wake word, test false activations in normal conversations, TV/background speech, and from different distances before lowering the cutoff.

### If the preview is wrong

Do **not** type `OK`. The notebook will stop safely before the expensive generation/training phase.

Change only `WAKE_WORD_PHONEMES` in the config cell, rerun from the configuration/Piper cells onward, and listen again.
